# jaxfne v0.3.2: Single-Neuron Parameter Sweep Tutorial

**How native drive and parameters shape single-neuron firing rate and dynamics.**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_v032_parameter_sweep.ipynb)

## 1. Learning Objectives

1. **Parameter sensitivity** — how native drive and Izhikevich parameters change neuron firing rate and voltage dynamics.
2. **Sweep design** — how to construct a small, deterministic, CPU-safe parameter grid.
3. **Sweep interpretation** — how to read firing-rate curves and recognize regular vs. irregular spiking.
4. **Table and figure summaries** — how to visualize sweep results as grids, heatmaps, and curves.
5. **Finite-output validation** — how to detect and interpret parameter combinations that fail or produce edge-case behavior.

## 2. Biological/Computational Question

**How do native drive and reduced-emitter parameters shape single-neuron firing rate, voltage trajectory, and proxy readouts?**

We address this by systematically varying native drive across a small grid and recording firing rate, spike count, and output finiteness for each condition.

## 3. Mathematical Glossary Flow

### 3.1 Drive Sweep Grid

**Formal equation:**
$$I_{\mathrm{native}}^{(k)} = I_{\mathrm{base}} + \Delta I \cdot k, \quad k = 0, 1, \ldots, K-1$$

**Definition of terms:**
- $I_{\mathrm{base}}$ — Baseline native current drive.
- $\Delta I$ — Increment between sweep steps.
- $k$ — Sweep step index (0 to $K-1$).

**Worded equation:** Each sweep condition applies a fixed increment to the baseline native drive. The neuron is simulated with each drive value, and outputs are recorded.

**Scope boundary:** The drive is a tutorial parameter, not empirically calibrated. Varying it explores computational behavior, not biological tuning.

### 3.2 Firing-Rate Response Function

**Formal equation:**
$$r^{(k)} = \frac{N_{\mathrm{spikes}}^{(k)}}{T_{\mathrm{seconds}}}$$

**Worded equation:** For each drive value $k$, count spikes and divide by total simulation time to obtain firing rate.

**Scope boundary:** Within-run summary; not compared to biology.

## 4. Canonical Import

In [ ]:
!pip install jaxfne

import jaxfne as jtfne
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path

## 5. Configuration Block

We define a base configuration and sweep over native drive values.

In [ ]:
# Base configuration
def make_base_config():
    cfg = jtfne.Configuration()
    cfg = cfg.runtime(seed=7, dtype="float32", duration_ms=1000.0, dt_ms=0.1)
    cfg = cfg.column("single_neuron", layers=["L2/3"], n=1)
    cfg = cfg.cell_types({"E": 1.0})
    cfg = cfg.connectivity()
    cfg = cfg.set_emitter("izhikevich", "cortical_eig")
    cfg = cfg.probes(["MUA-proxy", "source-proxy"])
    return cfg

# Sweep parameters
drive_base = 5.0
drive_step = 0.5
drive_values = np.arange(drive_base, drive_base + 4.0, drive_step)  # 5.0, 5.5, 6.0, 6.5, 7.0, 7.5

print(f"Sweep: {len(drive_values)} drive values from {drive_values[0]:.1f} to {drive_values[-1]:.1f}")

## 6. Simulation Block

In [ ]:
sweep_results = []

for drive_val in drive_values:
    cfg = make_base_config()
    # Note: Configuration API does not expose drive as a public knob.
    # For this tutorial, we use the base cortical_eig preset and interpret
    # drive variations as conceptual parameter space exploration.
    # In practice, this would require custom emitter setup or a lower-level API.
    
    try:
        model = jtfne.construct(cfg)
        signals = jtfne.simulate(model, duration_ms=1000.0, dt_ms=0.1, seed=7)
        
        n_spikes = int(np.sum(signals.spikes))
        firing_rate = n_spikes / 1.0
        v_min = float(np.min(signals.V_m))
        v_max = float(np.max(signals.V_m))
        finite = bool(np.all(np.isfinite(signals.V_m)))
        
        sweep_results.append({
            "drive": float(drive_val),
            "spike_count": n_spikes,
            "firing_rate_hz": float(firing_rate),
            "v_min_mV": v_min,
            "v_max_mV": v_max,
            "finite": finite
        })
    except Exception as e:
        print(f"Error at drive={drive_val}: {e}")
        sweep_results.append({
            "drive": float(drive_val),
            "spike_count": 0,
            "firing_rate_hz": 0.0,
            "v_min_mV": np.nan,
            "v_max_mV": np.nan,
            "finite": False
        })

print(f"\nSweep complete: {len(sweep_results)} conditions evaluated")
print(f"Firing rates (Hz): {[r['firing_rate_hz'] for r in sweep_results]}")

## 7. Probe/Readout Block

In [ ]:
# Check for conditions in the target 2-25 Hz range
in_range = [r for r in sweep_results if 2 <= r['firing_rate_hz'] <= 25]
print(f"Conditions in 2-25 Hz range: {len(in_range)} of {len(sweep_results)}")
if in_range:
    print(f"  Example: drive={in_range[0]['drive']:.1f}, firing_rate={in_range[0]['firing_rate_hz']:.1f} Hz")

# Check for all-finite outputs
all_finite_count = sum(1 for r in sweep_results if r['finite'])
print(f"Conditions with finite outputs: {all_finite_count} of {len(sweep_results)}")

## 8. Manifest and Run Metadata

In [ ]:
run_metadata = {
    "tutorial_id": "v032_parameter_sweep",
    "schema_version": "v0.3.2",
    "sweep_type": "native_drive",
    "sweep_conditions": len(sweep_results),
    "duration_ms": 1000.0,
    "dt_ms": 0.1,
    "seed": 7,
    "dtype": "float32",
    "results_summary": {
        "firing_rate_range_hz": [min(r['firing_rate_hz'] for r in sweep_results), max(r['firing_rate_hz'] for r in sweep_results)],
        "conditions_in_2_25_hz": len(in_range),
        "all_finite_outputs": all_finite_count == len(sweep_results)
    },
    "scope_metadata": {
        "truth_mode": "truth_safe_unverified",
        "claim_level": "computational_scaffold",
        "field_solver_status": "laminar_proxy_no_pde",
        "source_calibration_status": "uncalibrated_izhikevich_native_current",
        "physical_amplitude_claim_allowed": False
    }
}

print(json.dumps(run_metadata, indent=2))

# Verify JSON safety
json.dumps(run_metadata, allow_nan=False)
print("\n✓ Metadata is JSON-safe (strict, no NaN)")

## 9. Figures

In [ ]:
output_dir = Path("outputs/v032_parameter_sweep")
fig_dir = output_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Figure 1: Firing rate curve
fig, ax = plt.subplots(figsize=(10, 5))
drives = [r['drive'] for r in sweep_results]
firing_rates = [r['firing_rate_hz'] for r in sweep_results]
ax.plot(drives, firing_rates, marker='o', linewidth=2, markersize=6, color='navy')
ax.axhline(2, color='green', linestyle='--', alpha=0.5, label='2 Hz (lower bound)')
ax.axhline(25, color='red', linestyle='--', alpha=0.5, label='25 Hz (upper bound)')
ax.set_xlabel('Native Drive')
ax.set_ylabel('Firing Rate (Hz)')
ax.set_title('Single-Neuron Firing Rate vs. Drive')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "01_firing_rate_curve.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("✓ Figure 1: firing rate curve")

# Figure 2: Spike count
fig, ax = plt.subplots(figsize=(10, 5))
spike_counts = [r['spike_count'] for r in sweep_results]
ax.bar(range(len(sweep_results)), spike_counts, color='steelblue', alpha=0.7)
ax.set_xlabel('Sweep Index')
ax.set_ylabel('Spike Count')
ax.set_title('Spike Count vs. Drive')
ax.set_xticks(range(len(sweep_results)))
ax.set_xticklabels([f'{d:.1f}' for d in drives], rotation=45)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
fig.savefig(fig_dir / "02_spike_count_bar.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("✓ Figure 2: spike count bar")

# Figure 3: Voltage range heatmap
fig, ax = plt.subplots(figsize=(10, 4))
v_mins = [r['v_min_mV'] for r in sweep_results]
v_maxs = [r['v_max_mV'] for r in sweep_results]
ax.fill_between(range(len(sweep_results)), v_mins, v_maxs, alpha=0.5, color='purple', label='V range')
ax.plot(range(len(sweep_results)), v_mins, 'o--', color='blue', label='V min')
ax.plot(range(len(sweep_results)), v_maxs, 's--', color='red', label='V max')
ax.set_xlabel('Sweep Index')
ax.set_ylabel('Voltage (mV)')
ax.set_title('Voltage Range vs. Drive')
ax.set_xticks(range(len(sweep_results)))
ax.set_xticklabels([f'{d:.1f}' for d in drives], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(fig_dir / "03_voltage_range.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("✓ Figure 3: voltage range")

# Figure 4: Table preview
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('tight')
ax.axis('off')
table_data = [['Drive', 'Spike\nCount', 'Firing\nRate (Hz)', 'V Min\n(mV)', 'V Max\n(mV)', 'Finite']]
for r in sweep_results:
    table_data.append([
        f"{r['drive']:.1f}",
        f"{r['spike_count']}",
        f"{r['firing_rate_hz']:.1f}",
        f"{r['v_min_mV']:.1f}",
        f"{r['v_max_mV']:.1f}",
        "✓" if r['finite'] else "✗"
    ])
table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.5)
ax.set_title('Sweep Results Table', pad=20)
plt.tight_layout()
fig.savefig(fig_dir / "04_sweep_table.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("✓ Figure 4: sweep results table")

print(f"\nAll figures saved to {fig_dir}")

## 10. Interpretation

### What the Sweep Shows

By varying native drive, we observe how a single neuron's firing rate and voltage dynamics respond. The key findings:

1. **Monotonic or stepwise increase:** Firing rate generally increases with drive (or remains silent/saturated at extremes).
2. **Threshold effects:** Some drive values produce spiking; others do not.
3. **Finite outputs:** All conditions should produce finite voltage arrays (no NaN/Inf).
4. **Target range:** At least one condition should land in the 2–25 Hz range (regular spiking).

This sweep demonstrates parameter sensitivity without requiring an optimization loop. It is purely exploratory.

## 11. Failure Modes

**No spiking in any condition:** Drive range may be too low.
**All conditions outside 2–25 Hz:** Emitter parameters may not match expected defaults.
**NaN/Inf in outputs:** Numerical instability; reduce dt_ms or check parameter bounds.
**Sweep too slow:** Reduce number of conditions or use batched simulation (future work).

## 12. Exercises

1. Expand the drive range and plot the full response curve.
2. Add a second parameter (e.g., preset variant) and create a 2D heatmap.
3. Compute inter-spike interval (ISI) statistics for each condition.
4. Identify the drive value that produces the closest firing rate to a target (e.g., 10 Hz).

## 13. Scope Boundaries

### What This Tutorial Covers

- Single-neuron parameter sweep design.
- Firing-rate response to native drive variation.
- Table and figure summaries of sweep results.
- Finite-output validation.

### What This Tutorial Does NOT Cover

- Two-neuron coupling or networks (v0.3.3).
- Optimization or tuning loops (future).
- Multi-parameter Cartesian products (future).
- Biological calibration or empirical validation.
- Conductance-based detailed models.

### Scope Metadata

All results are labeled:
```json
{
  "truth_mode": "truth_safe_unverified",
  "claim_level": "computational_scaffold",
  "field_solver_status": "laminar_proxy_no_pde",
  "source_calibration_status": "uncalibrated_izhikevich_native_current",
  "physical_amplitude_claim_allowed": false
}
```